# 95662 - Introduction to Machine Learning
##### By: Brenno, Camilla, Kok Soon

## 0. Config and Imports

In [1]:
# !pip install requirements.txt

In [2]:
import os
from pathlib import Path
import json
import pandas as pd

In [3]:
# CODE TO DOWNLOAD DATA HERE (If using colab)

In [4]:
DATA_DIR = Path('./data')# Path to dataset

## 1. Data

### 1.1 File Sanity Check

In [5]:
expected_files = [
    "train.jsonl",
    "validation.jsonl",
    "test_id.jsonl",
    "test_ood.jsonl",
    "test_long.jsonl",
]

for filename in expected_files:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"

print("All dataset files found:")
for filename in expected_files:
    print(" -", DATA_DIR / filename)

All dataset files found:
 - data\train.jsonl
 - data\validation.jsonl
 - data\test_id.jsonl
 - data\test_ood.jsonl
 - data\test_long.jsonl


### 1.2 Data Loading

In [6]:
# Function to load jsonl into pandas df object
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

### 1.3 Understanding Data

#### 1.3.1 Data shapes

In [7]:
train_df = load_jsonl(DATA_DIR / "train.jsonl")
validation_df = load_jsonl(DATA_DIR / "validation.jsonl")
test_id_df = load_jsonl(DATA_DIR / "test_id.jsonl")
test_ood_df = load_jsonl(DATA_DIR / "test_ood.jsonl")
test_long_df = load_jsonl(DATA_DIR / "test_long.jsonl")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test ID:", test_id_df.shape)
print("Test OOD:", test_ood_df.shape)
print("Test long:", test_long_df.shape)

Train: (12000, 6)
Validation: (2000, 6)
Test ID: (2000, 6)
Test OOD: (2000, 6)
Test long: (1500, 6)


#### 1.3.2 Data Example

In [8]:
train_df.head()

,id,expression,value,length,operator_count,depth
0,train-00000,4+4+7+3+8-1,25,11,5,5
1,train-00001,5-(6-5)+8+8,20,11,4,5
2,train-00002,5-3+1,3,5,2,3
3,train-00003,1-(1+5+8+7+1),-21,13,5,6
4,train-00004,7+1-(6+5+9),-12,11,4,4


#### 1.3.3 Test for White Space

It is important to test for white spaces to know if it is required to strip them away because whitespaces such as (1 + 1 3) is a wrong mathematical expression. Furthermore, our plan to use a binary tree to linearize mathematical expression may run into issues if we use whitespace to separate terms

In [27]:
def check_for_whitespace(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    return combined_df["expression"].astype(str).str.contains(r"\s").any()

In [29]:
print("White Space Exist") if check_for_whitespace(train_df,validation_df,test_id_df,test_ood_df,test_long_df) else print("No Whitespace found")

No Whitespace found


#### 1.3.4 Test for Unary Operators

Unary operators like -5 have to be considered when constructing expression trees

In [57]:
def check_for_unary_operators(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    def has_unary(expr):
        for i, ch in enumerate(expr):
            if ch in "+-":
                if i == 0:
                    return True  # starts with + or -
                prev = expr[i - 1]
                if prev in "(-+":
                    return True
        return False

    return combined_df["expression"].apply(has_unary).any()

In [58]:
print("Unary Operator Exist") if check_for_whitespace(train_df,validation_df,test_id_df,test_ood_df,test_long_df) else print("No Unary Operator found")

No Unary Operator found


#### 1.3.5 Data Preprocessing and Transformation

In this step, we aim to represent our input data in a different form in an attempt to help our models learn better

In [16]:
# Node Structure (Intermediary structure required to build trees)
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

In [17]:
# ExpressionTree Class (class used to represent mathematical expression)
class ExpressionTree:
    def __init__(self, root_node):
        self.root_node = root_node

    def get_root_node(self):
        return self.root_node

    def to_postfix(self):
        def dfs(node):
            if node is None:
                return []
            return dfs(node.left) + dfs(node.right) + [str(node.value)]

        return " ".join(dfs(self.root_node))

    def to_prefix(self):
        def dfs(node):
            if node is None:
                return []
            return [str(node.value)] + dfs(node.left) + dfs(node.right)

        return " ".join(dfs(self.root_node))

In [18]:
def reduce(operator_stack, operand_stack):
    op = operator_stack.pop()

    right = operand_stack.pop()
    left = operand_stack.pop()

    operand_stack.append(Node(op, left, right))


def build_tree_with_expression(expression):
    subtree_stack = []
    operator_stack = []

    for ch in expression:

        if ch.isnumeric():
            subtree_stack.append(Node(ch))

        elif ch == '(':
            operator_stack.append(ch)

        elif ch == ')':
            while operator_stack[-1] != '(':
                reduce(operator_stack, subtree_stack)

            operator_stack.pop()  # remove '('

        elif ch in '+-':

            while (
                operator_stack
                and operator_stack[-1] != '('
            ):
                reduce(operator_stack, subtree_stack)

            operator_stack.append(ch)

    while operator_stack:
        reduce(operator_stack, subtree_stack)

    root = subtree_stack.pop()
    return ExpressionTree(root)

expression = "5-(6-5)+8+8"
build_tree_with_expression(expression).to_postfix()

'5 6 5 - - 8 + 8 +'

#### 1.3.6 Enriching Data

In this step, we aim to introduce more statistics to help us understand the data better.

#### 1.3.7 Summary of Data

Here, we create a function to give a summary of our data so we can compare the different data splits

In [46]:
# Function to return summary of data 
# It is possible to calculate the summary of multiple data split together by passing them in together
def summary(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    result = pd.DataFrame({
        "mean": combined_df.mean(numeric_only=True),
        "min": combined_df.min(numeric_only=True),
        "max": combined_df.max(numeric_only=True),
    })

    # Expecting comparison of data split with different size, so we have to normalise for more meaningful comparison
    result.loc["value", "positive_ratio"] = (combined_df["value"] > 0).mean()
    result.loc["value", "negative_ratio"] = (combined_df["value"] < 0).mean()
    result.loc["value", "zero_ratio"] = (combined_df["value"] == 0).mean()

    mode = combined_df.mode(numeric_only=True).iloc[0]
    result["mode"] = mode
    
    return result

In [47]:
summary(train_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.860750,-36,46,0.65725,0.306167,0.036583,2
length,9.607000,5,19,NaN,NaN,NaN,7
operator_count,3.489917,2,5,NaN,NaN,NaN,4
depth,3.689917,3,6,NaN,NaN,NaN,4


In [48]:
summary(validation_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.585,-32,47,0.64,0.323,0.037,4.0
length,9.604,5,19,NaN,NaN,NaN,7.0
operator_count,3.497,2,5,NaN,NaN,NaN,3.0
depth,3.679,3,6,NaN,NaN,NaN,3.0


In [49]:
summary(test_id_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,5.2325,-29,40,0.668,0.296,0.036,8
length,9.6090,5,19,NaN,NaN,NaN,7
operator_count,3.4870,2,5,NaN,NaN,NaN,2
depth,3.7220,3,6,NaN,NaN,NaN,4


In [50]:
summary(test_ood_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.8555,-38,44,0.638,0.332,0.03,5
length,12.4840,7,23,NaN,NaN,NaN,9
operator_count,4.5155,3,6,NaN,NaN,NaN,5
depth,4.4345,3,7,NaN,NaN,NaN,4


In [51]:
summary(test_long_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.024667,-42,51,0.609333,0.362,0.028667,7
length,17.744000,15,27,NaN,NaN,NaN,15
operator_count,6.214667,5,7,NaN,NaN,NaN,7
depth,5.500667,4,8,NaN,NaN,NaN,4
